In [1]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Step 1a - Indexing (Document Ingestion)

In [2]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

video_id = "C0gErQtnNFE"  # only the ID, not full URL

ytt_api = YouTubeTranscriptApi()

try:
    transcript_list = ytt_api.fetch(video_id, languages=["en"])

    # New API: chunk is an object, so use chunk.text
    transcript = " ".join(chunk.text for chunk in transcript_list)

    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

Something's obviously not quite right about the definition of intelligence. >> If we play this out, what's the limit here? >> The best use case of AI was to improve uh human health. It was the moment I've been waiting for that could achieve something no other system could. [music] I want to use AI as a tool to help us understand the nature of reality around it. >> Governments are going to use AI. What would you hope that they use it [music] for? There's two things to worry about. One is That's Demis Hassabis, the CEO of Google DeepMind. >> Nobel Prize winner. He is one of the most important people alive on what is quickly becoming the biggest technological leap in our lifetime. Because the biggest way that AI is going to impact our lives isn't something that we can see. It's not a chatbot, it's not an image generator. It's tools that are invisible to us in drug design and natural disaster detection and nuclear fusion and quantum computing. [music] Tools that he and his team are buildin

In [3]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text="Something's obviously not quite right", start=0.0, duration=3.44), FetchedTranscriptSnippet(text='about the definition of intelligence.', start=1.8, duration=3.56), FetchedTranscriptSnippet(text=">> If we play this out, what's the limit", start=3.44, duration=2.36), FetchedTranscriptSnippet(text='here?', start=5.36, duration=3.12), FetchedTranscriptSnippet(text='>> The best use case of AI was to improve', start=5.8, duration=4.68), FetchedTranscriptSnippet(text="uh human health. It was the moment I've", start=8.48, duration=3.84), FetchedTranscriptSnippet(text='been waiting for that could achieve', start=10.48, duration=3.44), FetchedTranscriptSnippet(text='something no other system could. [music]', start=12.32, duration=4.04), FetchedTranscriptSnippet(text='I want to use AI as a tool to help us', start=13.92, duration=4.2), FetchedTranscriptSnippet(text='understand the nature of reality around', start=16.36, duration=2.08), Fe

In [4]:
print(len(transcript_list))

1935


Step 1b - Indexing (Text Splitting)

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=70)
chunks = splitter.create_documents([transcript])

In [6]:
len(chunks), type(chunks)

(206, list)

In [7]:
chunks[10]

Document(metadata={}, page_content="advance science and medicine and I've [music] always thought of AI as potentially the ultimate tool to do that. So, I'm hoping we're going to talk about that today [music] and really that's been my passion for what to apply AI to. But of course it can be applied to many things. [music] Oh, this is going to be a lot of fun. Yeah. So, in this Jenga game that we have, a lot of these are blocks that")

Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [8]:
embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

# Create a FAISS vector store from the documents
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

In [10]:
vectorstore.index_to_docstore_id

{0: 'b47050da-d84c-408d-a05e-50b4ec7354de',
 1: '45e93058-6bdd-4d70-b301-4c903d4fdafc',
 2: 'b88be58a-f984-4363-87c0-3cc3ee7ac1c0',
 3: 'bded965c-3a9e-4ea8-9cf2-c59fb511238d',
 4: '44314b37-5404-4a8c-bba9-99b0ece08b4b',
 5: 'e4d1c455-5422-4fcb-82f7-0f13166134ef',
 6: 'eb5cd305-e5da-44e1-9caa-55e2b102b8cd',
 7: 'd7a3b375-30b5-4576-9152-a41abcbb166b',
 8: '295bf29c-3c16-4e8d-aad6-fc13283103f3',
 9: 'fa8fc1d4-2a45-4ac4-9e92-5543bfbc5f35',
 10: '59c62593-2bae-4c78-8796-859a93ff358f',
 11: 'cb2023ed-9d0e-4008-939a-19cf50eef27b',
 12: '7b052681-2c49-4a0d-aca0-83f030051dbb',
 13: '899d3d03-cfd8-4136-8bbe-1f7d84ba1a04',
 14: '16c9c712-645a-4078-8a4f-e6db8939ecbb',
 15: '44cb304f-a2b8-4b07-beee-959b1967d518',
 16: '27c6d43c-f874-47bb-8e4b-e203d0ed547e',
 17: 'f47077cf-67dc-4ed1-9254-a4be3679c024',
 18: '6f1b1770-65fb-44de-8ac7-ebab7c66881a',
 19: '8577f044-3941-486c-9887-0b72bedaac50',
 20: 'e6b776d2-4dd7-4cfe-b89d-3ae66d11563c',
 21: 'f381cf0b-f40d-4b3c-a033-6a29d0fd3e45',
 22: '11a7261c-ae26-

In [11]:
vectorstore.get_by_ids(['f381cf0b-f40d-4b3c-a033-6a29d0fd3e45'])

[Document(id='f381cf0b-f40d-4b3c-a033-6a29d0fd3e45', metadata={}, page_content="about setting up a system where scientists could send in a request for a specific protein, like a website, and then get the protein folded. Yes. And then someone else has a very different idea. Yes. Can you walk me through what happens in that meeting? [laughter] And then you your reaction is incredible uh and I really want to know what you were thinking. >> Yeah. Sure. Well, look, we we it's it")]

Step 2 - Retrieval

In [14]:
retriever = vectorstore.as_retriever(search_type = "similarity", search_kwargs={"k":4})

In [15]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7d1d3e5249d0>, search_kwargs={'k': 4})

In [16]:
retriever.invoke("What is th topic about ?")

[Document(id='d6ee885c-a258-4266-8399-777e47e86537', metadata={}, page_content="questions I came in for you with, you know, if I get an hour with you in my life, was next time I read a headline, how do I weight the concerns that we're all going to have over the next 30 years? You know, like what are the things that people are worrying too much about, and what are the things that they are not worrying en- enough about? >> Yeah. So, I think the two things I just mentioned are"),
 Document(id='c2e95d2e-ce80-408b-8bad-8a0bd8921273', metadata={}, page_content='about? >> Yeah. So, I think the two things I just mentioned are the things that maybe the average person is not worrying enough about, but even I think some of the experts and the scientists in the field. I feel like those are the key things that are more societal affecting, um, that if we if we There are other things that we need to worry about, too, like deepfakes and and we and we try to help'),
 Document(id='ae575461-8173-441f-80d

## Step 3 - Agumentation

In [18]:
# Here we form the LLM

llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0
)

In [19]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [27]:
question          = "is the topic of medicines and neuroscience discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [28]:
retrieved_docs

[Document(id='e6b776d2-4dd7-4cfe-b89d-3ae66d11563c', metadata={}, page_content="had been called one of the most important unsolved problems in modern medicine. >> Mhm. And it's 2021. >> Mhm. You're in a meeting. Mhm. I am so glad that there was a camera in this meeting because it is one of the most incredible moments I have ever seen. Can we use AlphaFold to to solve it? I think you're talking with your team about setting up a system where scientists could send in a request"),
 Document(id='df72f53a-97ec-4543-b00c-e932aa4b8545', metadata={}, page_content="Turing machine. So, the question is and but there are others like friends of mine like Roger Penrose and who you know, believes there might be some quantum effect in the brain, right? And I'm sure you probably done videos about it that that and he you know, we've had some very good-natured debates about this. But so far neuroscience hasn't found any quantum effects in the brain. Doesn't mean they"),
 Document(id='0f4918de-f161-43c8-b7

In [29]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"had been called one of the most important unsolved problems in modern medicine. >> Mhm. And it's 2021. >> Mhm. You're in a meeting. Mhm. I am so glad that there was a camera in this meeting because it is one of the most incredible moments I have ever seen. Can we use AlphaFold to to solve it? I think you're talking with your team about setting up a system where scientists could send in a request\n\nTuring machine. So, the question is and but there are others like friends of mine like Roger Penrose and who you know, believes there might be some quantum effect in the brain, right? And I'm sure you probably done videos about it that that and he you know, we've had some very good-natured debates about this. But so far neuroscience hasn't found any quantum effects in the brain. Doesn't mean they\n\nthese types of technologies should be able to help um across almost every therapeutic area. In prep for this, I did a background interview with your fellow Nobel Prize winner, John Jumper. He re

In [30]:
final_prompt = prompt.invoke({"context":context_text, "question": question })

In [31]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      had been called one of the most important unsolved problems in modern medicine. >> Mhm. And it's 2021. >> Mhm. You're in a meeting. Mhm. I am so glad that there was a camera in this meeting because it is one of the most incredible moments I have ever seen. Can we use AlphaFold to to solve it? I think you're talking with your team about setting up a system where scientists could send in a request\n\nTuring machine. So, the question is and but there are others like friends of mine like Roger Penrose and who you know, believes there might be some quantum effect in the brain, right? And I'm sure you probably done videos about it that that and he you know, we've had some very good-natured debates about this. But so far neuroscience hasn't found any quantum effects in the brain. Doesn't mean they\n\nthes

## Generations

In [32]:
answer = llm.invoke(final_prompt)

In [34]:
answer

AIMessage(content='The topic of medicines and neuroscience is not explicitly mentioned in this transcript.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-05-25T17:39:41.826807557Z', 'done': True, 'done_reason': 'stop', 'total_duration': 15001391066, 'load_duration': 275237174, 'prompt_eval_count': 436, 'prompt_eval_duration': 13170001577, 'eval_count': 15, 'eval_duration': 1521709548, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--019e6038-5be3-72f2-b207-7e3c18684ddd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 436, 'output_tokens': 15, 'total_tokens': 451})

## BUilding Chains

In [35]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [36]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [38]:
parallel_chain = RunnableParallel(
    {
        "context" : retriever | RunnableLambda(format_docs),
        "question" : RunnablePassthrough()
    }
)

In [39]:
parallel_chain.invoke("What the video about ?")

{'context': "battle in recent history, Demis is now in charge of much, much more. He's now behind basically everything Google does in AI. He's making decisions that affect your life and millions of other lives every single day. So, what is he planning to do with all of that power? My goal is to show you the future that Demis Hassabis wants [music] to build so that you can decide for yourself what you think of\n\nyeah. Well, so my hope in this conversation is to make this explainer together and [music] to help people see what's happening right now in AI really and what is the future that you see coming? What are you hoping to do in this conversation? Yeah, a lot of the reasons that I got into AI 30-plus years ago now is to um advance science and medicine and I've [music] always thought of AI as\n\nhad been called one of the most important unsolved problems in modern medicine. >> Mhm. And it's 2021. >> Mhm. You're in a meeting. Mhm. I am so glad that there was a camera in this meeting be

In [40]:
parser = StrOutputParser()

In [41]:
main_chain = parallel_chain | prompt | llm | parser

In [43]:
main_chain.invoke("Can you please summarize the video ?")

"I don't know what the video is referring to."